In [1]:
import torch
from torch import nn
from torch.distributions import Categorical
from environment import Environment
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
num_actions = 50  # Assuming your model has 50 outputs
action_range = np.linspace(-2, 2, num_actions)

In [4]:
actor = torch.load('./models/mlp_pid.pth') # model takes: 'target_lataccel', 'current_lataccel', 'vEgo', 'aEgo', 'roll', 'error', 'prev_error', 'integral', 'derivative', 'prev_derivative', 'prev_action'
actor.to(device)
actor.eval()

PidMLP(
  (fc1): Linear(in_features=11, out_features=128, bias=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=128, bias=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=128, out_features=128, bias=True)
  (dropout3): Dropout(p=0.3, inplace=False)
  (fc4): Linear(in_features=128, out_features=50, bias=True)
)

In [5]:
env = Environment()
state = env.reset()
state = torch.tensor(state, dtype=torch.float32, device=device)

integral = 0
prev_error = 0
prev_action = 0
prev_derivative = 0

while True:
    error = state[0] - state[1]
    integral += error
    derivative = error - prev_error

    with torch.no_grad():
        logits = actor(torch.tensor([[*state, error, prev_error, integral, derivative, prev_derivative, prev_action]], device=device, dtype=torch.float32))

    probs = nn.functional.softmax(logits, dim=-1)

    distribution = Categorical(probs)
    action_index = distribution.sample()

    action_index = torch.argmax(logits)

    action = action_range[action_index]

    next_state, cost, done = env.step(action)

    state = torch.tensor(next_state, dtype=torch.float32, device=device)

    prev_error = error
    prev_derivative = derivative
    prev_action = action

    print(f"Action: {action} Reward: {cost}")

    if done:
        print('Total Cost:', env.get_total_cost())
        break
        

Action: -0.04081632653061229 Reward: [ 0.1749068  16.350931   17.22546502]
Action: -0.04081632653061229 Reward: [7.73645349e-03 2.38884923e+01 2.39271746e+01]
Action: -0.04081632653061229 Reward: [ 0.06488611 15.2886351  15.61306567]
Action: -0.04081632653061229 Reward: [ 0.20681428 46.82144499 47.85551637]
Action: -0.04081632653061229 Reward: [0.15733826 0.         0.78669132]
Action: -0.04081632653061229 Reward: [0.25063758 0.95553969 2.20872758]
Action: -0.04081632653061229 Reward: [ 1.20120872 23.88849234 29.89453594]
Action: -0.04081632653061229 Reward: [0.92891754 3.82215877 8.46674648]
Action: -0.04081632653061229 Reward: [ 0.38947079 15.2886351  17.23598903]
Action: -0.04081632653061229 Reward: [0.27464816 0.         1.37324082]
Action: -0.04081632653061229 Reward: [0.03472399 8.59985724 8.77347719]
Action: -0.04081632653061229 Reward: [ 0.29306856 23.88849234 25.35383515]
Action: -0.04081632653061229 Reward: [ 1.18687764 34.39942897 40.33381715]
Action: -0.04081632653061229 Re